# P6–P10 formal and structured evidence

## Scope and authority

This notebook covers bounded formal certificates, structured navigation/authority/learning/problem-solving evidence, and explicit failed or not-executed gates. A complete finite certificate can validate its finite object; it does not become a universal theorem or independent external validation.


## Theory, methodology, and algorithms

For a frozen finite domain $D$ and explicit certificate $C$, the executable obligation has the form

$$\operatorname{Verify}(D,C)=\bigwedge_{x\in D}\phi(x,C).$$

Completeness of the planned run additionally requires

$$n_{\mathrm{observed}}=n_{\mathrm{planned}},$$

and replay binding can be written schematically as

$$H(I,C,O)=h_{\mathrm{registered}}.$$

Each equality has a distinct role. Certificate verification does not supply external authority; hash identity does not prove scientific truth; and a prospective protocol with no execution has no empirical result.


In [ ]:
from pathlib import Path
import json
import sys


def find_visualization_root(start=Path.cwd()):
    """Find visualization/ whether Jupyter starts at the repo root or notebooks/."""
    start = start.resolve()
    candidates = [start / "visualization", start, *start.parents]
    for candidate in candidates:
        if candidate.name == "visualization" and (candidate / "data" / "derived" / "atlas.json").exists():
            return candidate
        nested = candidate / "visualization"
        if (nested / "data" / "derived" / "atlas.json").exists():
            return nested
    raise FileNotFoundError(
        "Could not find visualization/data/derived/atlas.json. "
        "Build the atlas from the repository root first."
    )


VIS_ROOT = find_visualization_root()
sys.path.insert(0, str(VIS_ROOT / "src"))
ATLAS_PATH = VIS_ROOT / "data" / "derived" / "atlas.json"
atlas = json.loads(ATLAS_PATH.read_text(encoding="utf-8"))


def as_rows(value):
    """Return normalized records without changing their scientific values."""
    if isinstance(value, list):
        return [row for row in value if isinstance(row, dict)]
    if isinstance(value, dict):
        return [row for row in value.values() if isinstance(row, dict)]
    return []


def first(row, *keys, default=None):
    for key in keys:
        if key in row and row[key] is not None:
            return row[key]
    return default


def paper_id(row):
    raw = str(first(row, "paper_id", "paper", "id", default="UNSCOPED"))
    return raw.replace("ORION-", "")


def exact_status(row):
    return str(first(row, "terminal", "status", "result_state", "authority", default="UNSPECIFIED"))


def numeric_value(row):
    value = first(row, "value", "observed", "count", default=None)
    return float(value) if isinstance(value, (int, float)) and not isinstance(value, bool) else None


paper_states = as_rows(atlas.get("paper_states", []))
metrics_by_paper = atlas.get("metrics", {})
metrics = as_rows(atlas.get("metric_records", []))
anomalies = as_rows(atlas.get("anomalies", []))
sources = as_rows(atlas.get("sources", []))
des_execution = as_rows(atlas.get("des_execution", []))
framework_mechanics = atlas.get("framework_mechanics", {})

print(f"Atlas: {ATLAS_PATH}")
print(
    f"Loaded {len(paper_states)} paper states, {len(metrics)} metrics, "
    f"{len(anomalies)} anomalies, {len(des_execution)} frozen DES rows and "
    f"{len(sources)} sources."
)


In [ ]:
import matplotlib.pyplot as plt
import textwrap
from matplotlib.colors import ListedColormap  # noqa: F401 -- used by heatmap notebooks

plt.rcParams.update({
    "figure.figsize": (10, 5.5),
    "axes.grid": True,
    "grid.alpha": 0.20,
    "font.size": 10,
})

STATE_COLORS = {
    "PASS": "#2e7d32",
    "SUPPORTED": "#2e7d32",
    "FAIL": "#c62828",
    "GATE_NOT_MET": "#c62828",
    "CANNOT_CHECK": "#ef6c00",
    "UNKNOWN": "#6a1b9a",
    "NOT_AUTHORITY": "#455a64",
    "NOT_EXECUTED": "#757575",
}


def state_color(text):
    upper = str(text).upper()
    for token, color in STATE_COLORS.items():
        if token in upper:
            return color
    return "#1565c0"


def human_label(value, width=18):
    # Wrap machine identifiers without changing canonical capitalization.
    cleaned = str(value).replace("_", " ").replace(":", " — ")
    return "\n".join(textwrap.wrap(cleaned, width=width, break_long_words=False))



def print_records(rows, fields, limit=30):
    """Small dependency-free table for exact atlas fields."""
    rows = list(rows)
    if not rows:
        print("No records match the current display selectors.")
        return
    widths = {
        field: min(
            48,
            max(len(field), *(len(str(first(row, field, default=""))) for row in rows[:limit])),
        )
        for field in fields
    }
    print(" | ".join(field.ljust(widths[field]) for field in fields))
    print("-+-".join("-" * widths[field] for field in fields))
    for row in rows[:limit]:
        print(" | ".join(str(first(row, field, default=""))[: widths[field]].ljust(widths[field]) for field in fields))
    if len(rows) > limit:
        print(f"... {len(rows) - limit} more record(s); change DISPLAY_LIMIT to inspect them.")


## Editable selectors and thresholds

`MIN_RECORDS` controls which evidence-presence columns are shown. It is a display threshold, not a scientific acceptance threshold.


In [ ]:
PAPERS = ["P6", "P7", "P8", "P9", "P10"]
MIN_RECORDS = 1
DISPLAY_LIMIT = 50


## Results: evidence-presence heatmap

The heatmap is binary: it records whether the atlas contains one or more rows of each exact metric name for each paper. It neither rescales nor compares scientific values.


In [ ]:
metric_names = sorted({str(first(row, "metric", "name", "metric_name", default="UNNAMED")) for row in metrics if paper_id(row) in PAPERS})
metric_names = [
    name for name in metric_names
    if sum(1 for row in metrics if paper_id(row) in PAPERS and str(first(row, "metric", "name", "metric_name", default="UNNAMED")) == name) >= MIN_RECORDS
]
presence = [
    [int(any(paper_id(row) == pid and str(first(row, "metric", "name", "metric_name", default="UNNAMED")) == name for row in metrics)) for name in metric_names]
    for pid in PAPERS
]

fig, ax = plt.subplots(figsize=(8.5, max(5.5, 0.45 * len(metric_names))))
if metric_names:
    ax.imshow(
        list(map(list, zip(*presence))),
        aspect="auto",
        cmap=ListedColormap(["#f5f5f5", "#00897b"]),
        vmin=0,
        vmax=1,
    )
    ax.set_xticks(range(len(PAPERS)), PAPERS)
    ax.set_yticks(range(len(metric_names)), [human_label(name, 28) for name in metric_names])
    ax.grid(False)
    ax.set_title("Metric-record presence (binary, not performance)")
else:
    ax.text(0.5, 0.5, "No metrics match the display threshold", ha="center", va="center")
    ax.set_axis_off()
plt.tight_layout()
plt.show()


In [ ]:
formal_states = [row for row in paper_states if paper_id(row) in PAPERS]
formal_anomalies = [row for row in anomalies if paper_id(row) in PAPERS]
print("EXACT STATES")
print_records(formal_states, ["paper_id", "title", "status", "terminal", "authority", "claim_ceiling"], DISPLAY_LIMIT)
print("\nANOMALIES")
print_records(formal_anomalies, ["paper_id", "anomaly_id", "severity", "status", "summary", "explanation"], DISPLAY_LIMIT)


## Discussion: anomalies and honest negatives

- **P7:** 738 cases were planned but only 736 were observed. The run is invalid; the residual is $738-736=2$, not a rounding issue or a near-pass.
- **P9:** the digits D-A result remains `CANNOT_CHECK`. A separately registered revival receipt preserves the append-only replay-failure terminal, records archive-matched replay agreement, and leaves the scientific successor frozen and unexecuted. Do not erase either the historical failure or the remaining scientific boundary.
- **P10:** the prospective study was not executed. Protocol existence, code, or historical results are not a substitute for prospective observations.

P6 finite-certificate evidence remains bounded to the explicit certified structures. P8 should be read only through its exact atlas status and source receipt.

## Claim ceiling

The heatmap demonstrates evidence inventory coverage, not algorithmic superiority. Formal/local evidence remains bounded; invalid, discrepant, and not-executed studies cannot be promoted.
